# 第4回：機械学習・深層学習で投稿を分類する

**このNotebookでできるようになること**

1. 教師あり学習：検索語をラベルにして「どの話題の投稿か」を当てる分類器を作り，精度を評価する
2. **自分でラベルを付けた少量のデータ**から分類器を作り，大量の投稿に自動でラベルを付ける（プロジェクトで最も使う道具）
3. 教師なし学習：ラベルなしで投稿をグループ分けする（クラスタリング）
4. 深層学習（YOLO）で投稿画像に写っている物体を検出し，テキストと組み合わせる

第3回で保存した `data/posts_words.csv` を使います（無ければサンプルデータを使います）．

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bsky_utils import *

pd.set_option("display.max_colwidth", 60)
set_japanese_font()

df = load_posts("data/posts_words.csv")
if "words_str" not in df.columns:                      # 第3回を飛ばした場合はここで形態素解析する
    df["words_str"] = df["text"].map(lambda s: " ".join(tokenize(s)))
df["words"] = df["words_str"].fillna("").str.split()
print(df["query"].value_counts())

## 1. 教師あり学習①：どの話題の投稿かを当てる

**教師あり学習**は「入力（本文）と正解（ラベル）」のペアから，入力→正解の対応を学ぶ方法です．
まずは，検索語 `query` を正解ラベルとみなして分類器を作ります（正解がすでにあるので，手順の練習に最適）．

手順：
1. 本文を数値ベクトルにする（**TF-IDF**：各単語の重要度を並べたベクトル）
2. データを **学習用** と **評価用** に分ける（覚えたデータで採点しないため）
3. 学習用で **ロジスティック回帰** を学習
4. 評価用で正解率を測る

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# ラベルが2種類以上あるデータを用意する（検索語が1つだけならサンプルデータで練習）
if df["query"].nunique() < 2:
    print("検索語が1つだけなので，サンプルデータで練習します．")
    df_sup = load_posts("sample_data/posts_sample.csv")
    df_sup["words_str"] = df_sup["text"].map(lambda s: " ".join(tokenize(s)))
else:
    df_sup = df.copy()

X_text = df_sup["words_str"].fillna("")
y = df_sup["query"]

X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.3, random_state=0, stratify=y)
print("学習用:", len(X_train), "件  評価用:", len(X_test), "件")

vec = TfidfVectorizer(token_pattern=r"(?u)\S+")      # 空白区切りの単語をそのまま使う
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
print("特徴（単語）の数:", len(vec.get_feature_names_out()))

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

pred = clf.predict(X_test_vec)
print("正解率:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))

In [ ]:
# 混同行列：どのラベルをどのラベルと間違えたか
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=clf.classes_), display_labels=clf.classes_).plot(ax=ax, colorbar=False)
plt.title("混同行列（行：正解，列：予測）")
plt.tight_layout()
plt.show()

In [ ]:
# 分類の根拠になった単語：各ラベルで係数が大きい単語 Top10
terms = np.array(vec.get_feature_names_out())
coef = clf.coef_ if len(clf.classes_) > 2 else np.vstack([-clf.coef_[0], clf.coef_[0]])
for label, row in zip(clf.classes_, coef):
    idx = row.argsort()[::-1][:10]
    print(f"[{label}] " + ", ".join(terms[idx]))

> 検索語そのものが本文に入っているので当てやすいのは当然です．重要なのは **「学習→評価→根拠の確認」という手順** と，評価用データを分ける理由を理解することです．

## 2. 教師あり学習②：自分でラベルを付けて分類する（プロジェクト用の道具）

プロジェクトで本当に知りたいのは，「防災の投稿のうち **要望** はどれくらいか」「**苦情**と**称賛**の比率は」のような，データに最初から付いていない分類です．

そのときは次の手順を使います．

1. 投稿を100〜200件ほど取り出して **人手でラベルを付ける**（Excelで `label` 列に記入）
2. それを学習データにして分類器を作る
3. 残りの数千件に **自動でラベルを付ける**
4. 精度（評価用データでの正解率）と一緒に結果を報告する

### 2-1. ラベル付け用シートを書き出す

`data/labeling_sheet.csv` をExcelで開き，`label` 列に分類名を入力して保存します（例：`情報提供` / `意見・要望` / `体験・感想` / `宣伝・告知`）．

In [ ]:
sheet = df.sample(n=min(150, len(df)), random_state=0)[["post_id", "query", "text"]].copy()
sheet["label"] = ""
sheet.to_csv("data/labeling_sheet.csv", index=False, encoding="utf-8-sig")
print("data/labeling_sheet.csv を書き出しました（", len(sheet), "件）．Excelで label 列を記入してください．")
sheet.head()

### 2-2. ラベル付きデータで学習する

自分たちのラベルがまだ無い場合は，授業用のラベル付きサンプル（`sample_data/labeled_sample.csv`，合成データ）で手順を確認します．

In [ ]:
own = pd.read_csv("data/labeling_sheet.csv", encoding="utf-8-sig")
own = own[own["label"].fillna("").astype(str).str.strip() != ""]

if len(own) >= 40 and own["label"].nunique() >= 2:
    labeled = own
    print("自分たちのラベル付きデータを使います:", len(labeled), "件")
else:
    labeled = pd.read_csv("sample_data/labeled_sample.csv", encoding="utf-8-sig")
    print("ラベルがまだ無いので，サンプルのラベル付きデータで練習します:", len(labeled), "件")

labeled["words_str"] = labeled["text"].map(lambda s: " ".join(tokenize(s)))
print(labeled["label"].value_counts())

In [ ]:
from sklearn.pipeline import make_pipeline

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    labeled["words_str"], labeled["label"], test_size=0.3, random_state=0, stratify=labeled["label"])

# ベクトル化と分類器を1つにまとめた「パイプライン」
model = make_pipeline(TfidfVectorizer(token_pattern=r"(?u)\S+"), LogisticRegression(max_iter=1000))
model.fit(Xl_train, yl_train)

pred_l = model.predict(Xl_test)
print("正解率:", round(accuracy_score(yl_test, pred_l), 3))
print(classification_report(yl_test, pred_l))

### 2-3. 全投稿に自動でラベルを付ける

In [ ]:
df["pred_label"] = model.predict(df["words_str"].fillna(""))
df["pred_prob"] = model.predict_proba(df["words_str"].fillna("")).max(axis=1).round(2)   # 予測の確信度

dist = df.groupby(["query", "pred_label"]).size().unstack(fill_value=0)
dist.plot(kind="bar", stacked=True, figsize=(7, 4))
plt.ylabel("投稿数")
plt.title("自動ラベルの分布（検索語別）")
plt.tight_layout()
plt.show()

df[["query", "text", "pred_label", "pred_prob"]].sort_values("pred_prob", ascending=False).head(10)

> **報告のしかた**：「分類器の正解率は評価用データで 0.85 だった．この分類器を全 2,300 件に適用したところ，要望は 18% だった」のように，**精度と一緒に**結果を示します．確信度（`pred_prob`）が低い投稿を目視で確認するのも有効です．

## 3. 教師なし学習：ラベルなしで投稿をグループ分けする（クラスタリング）

ラベルを付ける前に「そもそもどんな種類の投稿があるのか」を知りたいときは，**クラスタリング**が役立ちます．
ここでは K-means で投稿を `K` 個のグループに分け，各グループの特徴語を見ます．

In [ ]:
from sklearn.cluster import KMeans

K = 5                                                   # ← グループ数．3〜8くらいで変えてみる
vec_all = TfidfVectorizer(token_pattern=r"(?u)\S+", min_df=2)
X_all = vec_all.fit_transform(df["words_str"].fillna(""))

km = KMeans(n_clusters=K, random_state=0, n_init=10)
df["cluster"] = km.fit_predict(X_all)

terms_all = np.array(vec_all.get_feature_names_out())
for k in range(K):
    idx = km.cluster_centers_[k].argsort()[::-1][:8]
    n = int((df["cluster"] == k).sum())
    print(f"クラスタ{k}（{n}件）: " + ", ".join(terms_all[idx]))

In [ ]:
# 各クラスタの投稿例を見る（特徴語だけでは分からない中身を確認する）
for k in range(K):
    print(f"\n=== クラスタ{k} ===")
    for text in df[df["cluster"] == k]["text"].head(3):
        print(" -", text[:70])

In [ ]:
# クラスタと検索語の関係
pd.crosstab(df["cluster"], df["query"])

> クラスタは「機械が見つけた似たもの同士のまとまり」であって，意味のある分類とは限りません．**人が中身を読んで名前を付ける**ところまでが分析です．

## 4. 【参考】深層学習：投稿画像に何が写っているか（YOLOによる物体検出）

> **この節は参考です（授業の本編では扱いません）．** SNS の画像にはセンシティブな内容が含まれることがあるため，画像を扱うプロジェクトは教員に相談のうえ，参考資料 `reference/image_download.ipynb` の注意事項に従って画像を用意してください．画像が無ければこの節は自動的にスキップされます．

ここまでは文字の分析でした．SNSには画像も多く，「食べ物の写真が多い」「人物が写った投稿は反応が多い」のような分析ができます．

深層学習の物体検出モデル **YOLOv5** を使って，参考資料の手順でダウンロードした画像（`data/images/`）に写っている物体（人，車，犬，食べ物など80種類）を検出します．
学習済みモデル `yolov5s.onnx` は画像認識テーマと共通で，`../image_recognition/det/` に置きます（入手方法は `image_recognition/README.md`）．

In [ ]:
import os
import cv2                        # OpenCV（画像認識テーマと共通）

MODEL = "../image_recognition/det/yolov5s.onnx"
NAMES = "../image_recognition/det/coco.names"
IMAGE_DIR = "data/images"

if not os.path.exists(MODEL):
    print("学習済みモデルが見つかりません:", MODEL)
    print("image_recognition/README.md の手順で yolov5s.onnx を用意してください．")
else:
    print("モデルOK:", MODEL)

In [ ]:
# YOLOv5 (ONNX) を OpenCV で動かす関数（image_recognition/03_deep_learning.ipynb と同じ）
INPUT_SIZE = 640
CONF_THRESHOLD, SCORE_THRESHOLD, NMS_THRESHOLD = 0.45, 0.5, 0.45

def detect_objects(net, classes, img):
    # 画像(BGR)から物体を検出し，[(クラス名, 信頼度, (x, y, w, h)), ...] を返す
    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (INPUT_SIZE, INPUT_SIZE), swapRB=True, crop=False)
    net.setInput(blob)
    out = net.forward(net.getUnconnectedOutLayersNames())[0][0]
    boxes, confs, ids = [], [], []
    for row in out:
        conf = float(row[4])
        if conf < CONF_THRESHOLD:
            continue
        scores = row[5:]
        cid = int(np.argmax(scores))
        if scores[cid] < SCORE_THRESHOLD:
            continue
        cx, cy, bw, bh = row[:4]
        boxes.append([int((cx - bw / 2) * w / INPUT_SIZE), int((cy - bh / 2) * h / INPUT_SIZE),
                      int(bw * w / INPUT_SIZE), int(bh * h / INPUT_SIZE)])
        confs.append(conf)
        ids.append(cid)
    keep = cv2.dnn.NMSBoxes(boxes, confs, CONF_THRESHOLD, NMS_THRESHOLD)
    keep = [int(i) for i in np.array(keep).ravel()] if len(keep) else []
    return [(classes[ids[i]], confs[i], tuple(boxes[i])) for i in keep]

In [ ]:
results = []
if os.path.exists(MODEL) and os.path.exists(os.path.join(IMAGE_DIR, "images.csv")):
    net = cv2.dnn.readNet(MODEL)
    classes = open(NAMES, encoding="utf-8").read().strip().split("\n")
    images = pd.read_csv(os.path.join(IMAGE_DIR, "images.csv"))
    for _, row in images.iterrows():
        img = cv2.imread(row["image_path"])
        if img is None:
            continue
        for name, conf, box in detect_objects(net, classes, img):
            results.append({"post_id": row["post_id"], "object": name, "conf": round(conf, 2)})
    det = pd.DataFrame(results)
    print("検出結果:", len(det), "個")
    if len(det):
        det["object"].value_counts().head(15).plot(kind="barh", figsize=(6, 5))
        plt.title("投稿画像に写っていた物体（検出数）")
        plt.xlabel("検出数")
        plt.tight_layout()
        plt.show()
else:
    det = pd.DataFrame(columns=["post_id", "object", "conf"])
    print("モデルまたは画像が無いため，この節はスキップします．")

In [ ]:
# 検出結果を画像に描いて確認する（最初の4枚）
if len(results):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (_, row) in zip(axes, images.head(4).iterrows()):
        img = cv2.imread(row["image_path"])
        for name, conf, (x, y, w, h) in detect_objects(net, classes, img):
            cv2.rectangle(img, (x, y), (x + w, y + h), (255, 178, 50), 2)
            cv2.putText(img, f"{name} {conf:.2f}", (x, max(y - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# テキストと画像を組み合わせる：人物が写っている投稿と，そうでない投稿の反応を比べる
if len(det):
    photo_posts = load_posts("data/posts.csv") if os.path.exists("data/posts.csv") else df
    has_person = det[det["object"] == "person"]["post_id"].unique()
    photo_posts["has_person"] = photo_posts["post_id"].isin(has_person)
    display(photo_posts.groupby("has_person")["like_count"].agg(["count", "median", "mean"]).round(1))

## 5. 発展：もっと高度な深層学習（Google Colab で）

| やりたいこと | 方法 |
|---|---|
| ポジティブ／ネガティブの判定（感情分析） | Hugging Face の日本語BERT感情分析モデルを `transformers` で使う（GPUがあるColabが楽）．必ず一部を人手で確認して精度を測る |
| 自分たちの分類を高精度に | BERTを自分たちのラベル付きデータで微調整（fine-tuning） |
| 画像を自由な言葉で分類（「食べ物」「風景」など） | CLIPのゼロショット分類 |
| 独自の物体（例：ヘルメット）の検出 | 画像認識テーマの `advanced/yolo_custom_training.md` の手順でYOLOを学習 |

感情分析の最小コード例（Colab用）：
```python
!pip install transformers fugashi unidic-lite
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="＜Hugging Face で 'japanese sentiment' を検索して選んだモデル名＞")
clf(["非常食のアルファ米を試食してみたら意外とおいしかった。", "防災無線が全然聞こえない。"])
```

> 辞書で「良い語／悪い語」を数える単純な感情分析は，否定・皮肉・文脈に弱く，数値を過信しやすいので本教材では扱いません．使う場合は必ず**人手で採点した評価データ**で精度を確認してください．

## まとめ：3回の演習で手に入れた道具

| 段階 | 道具 |
|---|---|
| 第2回 データ取得 | キーワード・期間・アカウント・返信・画像の取得，CSV保存と取得条件の記録 |
| 第3回 加工・可視化 | 形態素解析，頻出語，ワードクラウド，時系列，共起ネットワーク，TF-IDF |
| 第4回 機械学習 | 教師あり分類（自分のラベルで学習→大量データに適用），クラスタリング（参考：YOLOによる投稿画像の物体検出） |

**プロジェクトの考え方**：「社会の誰の，どんな困りごとを，SNSのデータで見える化・改善できるか」を先に決め，その問いに答えるために上の道具を組み合わせてください．道具を全部使う必要はありません．